# Image Processing Fundamental

**Course:** Image Processing  
**Level:** M1 CORO / DASSIP  
**Prerequisites:** basic Python and NumPy only — no previous image-processing background is assumed.  
**Purpose:** build a complete mental model of digital images before image transformation, filtering, frequency-domain processing, and segmentation.

## Goal

This notebook is the entry point to image processing. It starts from first principles and treats an image as both a visual object and numerical data.

By the end, you should be able to:

1. explain how a real scene becomes a digital image through sampling and quantization;
2. describe pixels, coordinates, resolution, channels, data types, bit depth, and dynamic range;
3. distinguish binary, grayscale, and RGB images;
4. load, inspect, display, and save images reproducibly;
5. understand NumPy image indexing and the difference between `(row, column)` and `(x, y)`;
6. convert RGB images to grayscale from first principles;
7. read and modify individual pixels safely;
8. extract regions of interest and local pixel neighborhoods;
9. separate and interpret RGB channels;
10. compute descriptive image statistics and intensity histograms;
11. understand intensity normalization and safe numerical operations;
12. introduce controlled image noise and quantify its effect with MAE, MSE, RMSE, and PSNR;
13. recognize common implementation mistakes before moving to more advanced image-processing methods.

### How to use this notebook

Read the Markdown explanation **before** running the code below it. For every section, ask three questions:

- **What does the concept mean mathematically?**
- **How is it represented in NumPy?**
- **What changes visually when the numbers change?**

Do not memorize function names. The goal is to understand the data model well enough that later operations—filtering, segmentation, Fourier transforms—feel like operations on arrays rather than magic.

### The big picture

A typical image-processing workflow is:

[
	ext{scene} ightarrow 	ext{acquisition} ightarrow 	ext{digital image} ightarrow 	ext{processing} ightarrow 	ext{analysis / decision}
]

Examples of processing include contrast enhancement, denoising, edge extraction, geometric transformation, segmentation, and frequency-domain filtering.

Almost all of them depend on the same foundation developed here: **an image is a structured numerical array**.

## 0. Setup

The notebook uses only relative paths, NumPy, Matplotlib, and Pillow.

The code is intentionally explicit: important operations are written out so that the numerical representation of an image remains visible instead of being hidden behind high-level functions.

In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Reproducible random generator used later for synthetic noise.
RNG = np.random.default_rng(42)

LAB_DIR = Path.cwd().parent
IMAGE_PROCESSING_DIR = LAB_DIR.parent
DATA_DIR = IMAGE_PROCESSING_DIR / "data" / "common"
FIG_DIR = LAB_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "einstein": DATA_DIR / "einstein.png",
    "peppers": DATA_DIR / "peppers.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "grass": DATA_DIR / "grass.jpg",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
}

missing = [str(path) for path in IMAGE_FILES.values() if not path.exists()]
assert not missing, f"Missing input files: {missing}"

print("Data directory: ../../data/common")
print("Figure directory: ../outputs/figures")
print(f"Images available: {len(IMAGE_FILES)}")

Data directory: ../../data/common
Figure directory: ../outputs/figures
Images available: 5


## 1. What is a digital image?

A grayscale image can be represented as a 2-D function


$$I(x,y)$$

where each pixel stores one intensity value. In code, this becomes a 2-D NumPy array with shape `(height, width)`.

An RGB color image stores three values at each pixel:

$$I(x,y,c), \qquad c \in \{R,G,B\}$$

so the array shape is normally `(height, width, 3)`.


### From a real scene to a digital image

A physical scene is continuous in space and light intensity. A digital camera converts that continuous information into a finite grid of numbers.

Two operations are fundamental:

- **Sampling** discretizes spatial position. It decides *where* measurements are taken and therefore determines the pixel grid.
- **Quantization** discretizes intensity. It decides *which numerical levels* can represent each measurement.

If an 8-bit channel is used, there are

[
2^8 = 256
]

possible values, usually from 0 to 255.

A digital image is therefore discrete in both **position** and **intensity**.

### Main image types

| Image type | Typical NumPy shape | Pixel meaning |
|---|---:|---|
| Binary | `(H, W)` | two classes, commonly 0 and 1 or 0 and 255 |
| Grayscale | `(H, W)` | one intensity value per pixel |
| RGB color | `(H, W, 3)` | red, green, and blue values per pixel |

Later labs will introduce masks, labels, filtered images, spectra, and segmentation maps, but all of them are still arrays with a defined numerical meaning.

In [2]:
toy_image = np.array([
    [0, 40, 80, 120, 160],
    [20, 60, 100, 140, 180],
    [40, 80, 120, 160, 200],
    [60, 100, 140, 180, 220],
    [80, 120, 160, 220, 255],
], dtype=np.uint8)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(toy_image, cmap="gray", vmin=0, vmax=255)
ax.set_title("A 5×5 grayscale image is a matrix of intensities")
for row in range(toy_image.shape[0]):
    for col in range(toy_image.shape[1]):
        ax.text(col, row, str(toy_image[row, col]), ha="center", va="center", fontsize=8)
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_grayscale_matrix.png", dpi=100, bbox_inches="tight")
plt.show()

print("shape:", toy_image.shape)
print("dtype:", toy_image.dtype)
print("minimum / maximum:", toy_image.min(), toy_image.max())

shape: (5, 5)
dtype: uint8
minimum / maximum: 0 255


### Coordinate convention

In mathematics we often write a pixel as $I(x,y)$. NumPy arrays are indexed as `image[row, column]`, which corresponds to `image[y, x]`. Mixing these conventions is a common source of bugs.


### Image origin and axis direction

Most image libraries place the origin at the **top-left** corner.

- increasing column index moves to the **right**;
- increasing row index moves **downward**.

That is different from the usual Cartesian plane, where the vertical axis points upward.

For a NumPy image:

[
	ext{pixel at Cartesian-like }(x,y) quad longleftrightarrow quad 	ext{image}[y,x]
]

This convention is essential when cropping, drawing, tracking objects, or interpreting coordinates returned by computer-vision algorithms.

## 2. Load and inspect reference images

Pillow loads each file, and NumPy exposes its numerical representation.


In [3]:
# Load every input as RGB so later examples use a consistent H×W×3 representation.
images = {}
for name, path in IMAGE_FILES.items():
    pil_image = Image.open(path).convert("RGB")
    images[name] = np.asarray(pil_image)

for name, image in images.items():
    print(
        f"{name:9s} | shape={str(image.shape):16s} "
        f"dtype={image.dtype} range=[{image.min()}, {image.max()}]"
    )

einstein  | shape=(256, 256, 3)    dtype=uint8 range=[0, 255]
peppers   | shape=(417, 606, 3)    dtype=uint8 range=[0, 254]
ballons   | shape=(360, 500, 3)    dtype=uint8 range=[0, 255]
grass     | shape=(240, 247, 3)    dtype=uint8 range=[0, 255]
tower     | shape=(427, 437, 3)    dtype=uint8 range=[0, 255]


In [4]:
# Display the reference images side-by-side to compare size, content, and aspect ratio.
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")
fig.suptitle("Reference images used in the fundamental lab", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_dataset_overview.png", dpi=100, bbox_inches="tight")
plt.show()

## 3. Dimensions, resolution, channels, and bit depth

For an RGB array `image.shape == (H, W, 3)`:

- `H` is image height in pixels;
- `W` is image width in pixels;
- `3` is the number of channels;
- `uint8` stores integers from 0 to 255, so each channel uses 8 bits.

A 24-bit RGB pixel therefore contains three 8-bit channel values.


### Pixel dimensions are not the same as physical resolution

An image that is (1920 	imes 1080) contains 1920 columns and 1080 rows of pixels. That tells us the **pixel dimensions**, not the physical size of an object in meters or millimeters.

Physical resolution depends on the acquisition system, field of view, sensor geometry, calibration, and sometimes metadata such as pixel spacing.

In image-processing code, always distinguish:

- **width / height in pixels**;
- **physical dimensions**;
- **display size**;
- **file size on disk**.

These quantities are related, but they are not interchangeable.

### Data type and bit depth

The data type controls which values can be stored.

For `uint8`:

[
0 le I le 255
]

For floating-point processing, images are often converted to `float32` or `float64`, frequently with values scaled to ([0,1]).

A crucial rule is:

> **Convert before arithmetic when an operation may exceed the valid range, then clip before converting back.**

Otherwise unsigned integer arithmetic can overflow or underflow.

### A practical mental model for `uint8` and floating-point images

`uint8` is compact and convenient for storage/display. Floating-point arrays are usually safer for intermediate calculations.

Typical workflow:

```python
image_float = image_uint8.astype(np.float32) / 255.0
# processing...
result_uint8 = np.clip(result_float * 255.0, 0, 255).astype(np.uint8)
```

Never assume a function expects ([0,255]) or ([0,1]). Check the expected range.

In [5]:
# For an RGB image, shape is (height, width, channels).
peppers = images["peppers"]
height, width, channels = peppers.shape
bytes_in_array = peppers.nbytes

print(f"Width: {width} pixels")
print(f"Height: {height} pixels")
print(f"Channels: {channels}")
print(f"Pixels: {width * height:,}")
print(f"Array memory: {bytes_in_array:,} bytes ({bytes_in_array / 1024**2:.2f} MiB)")
print(f"One RGB pixel example at (y=100, x=200): {peppers[100, 200]}")

Width: 606 pixels
Height: 417 pixels
Channels: 3
Pixels: 252,702
Array memory: 758,106 bytes (0.72 MiB)
One RGB pixel example at (y=100, x=200): [164 196 232]


## 4. RGB to grayscale from first principles

A naive average treats all channels equally, but human vision is more sensitive to green than blue. A common luminance approximation is

$$Y = 0.299R + 0.587G + 0.114B.$$

The result is clipped and converted back to `uint8`.


### Why grayscale matters

Grayscale removes color information and keeps one intensity value per pixel. This reduces the amount of data and is sufficient for many tasks where structure or brightness matters more than color.

However, grayscale conversion is **not** simply "take one channel". A weighted luminance model combines RGB channels according to their contribution to perceived brightness.

In [6]:
def rgb_to_grayscale(rgb_image):
    """Convert an RGB uint8 image to grayscale using luminance weights."""
    rgb_float = rgb_image.astype(np.float32)
    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )
    return np.clip(gray, 0, 255).astype(np.uint8)

einstein_rgb = images["einstein"]
einstein_gray = rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(einstein_rgb)
axes[0].set_title("RGB")
axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Grayscale luminance")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_rgb_to_grayscale.png", dpi=100, bbox_inches="tight")
plt.show()

print("RGB shape:", einstein_rgb.shape)
print("Grayscale shape:", einstein_gray.shape)

RGB shape: (256, 256, 3)
Grayscale shape: (256, 256)


### Binary images and masks

A binary image contains only two logical states, such as foreground/background or object/not-object.

A common representation is:

[
B(x,y)=
egin{cases}
255 & 	ext{if } I(x,y)ge T\
0 & 	ext{otherwise}
end{cases}
]

where (T) is a threshold.

Binary images are especially important later for segmentation and morphology. At this stage, remember that a binary image is still a 2-D numerical array; only its allowed values are restricted.

## 5. Accessing and modifying pixels

Direct indexing is useful for understanding representation, but processing algorithms should normally operate on regions or complete arrays rather than Python loops over every pixel. Always edit a copy if the original image must be preserved.


### Pixel access: read first, modify second

Direct pixel editing is useful for learning, debugging, annotation, and small local operations.

Two good habits:

1. make a copy before modifying an image you still need;
2. keep row/column order explicit.

For large operations, vectorized NumPy expressions are usually clearer and much faster than nested Python loops.

In [7]:
# Work on a copy so the original image remains unchanged for later experiments.
edited = images["ballons"].copy()
y, x = 120, 250
original_pixel = edited[y, x].copy()

# Mark a small square in pure red around the selected pixel.
edited[y-5:y+6, x-5:x+6] = [255, 0, 0]

print("Original RGB value:", original_pixel)
print("Edited center RGB value:", edited[y, x])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(images["ballons"])
axes[0].scatter([x], [y], s=80, facecolors="none", edgecolors="yellow", linewidths=2)
axes[0].set_title("Original + selected pixel")
axes[1].imshow(edited)
axes[1].set_title("Edited copy")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_pixel_edit.png", dpi=100, bbox_inches="tight")
plt.show()

Original RGB value: [243 218 161]
Edited center RGB value: [255   0   0]


### Pixel neighborhoods

Many image-processing methods do not use a pixel alone. They use a **neighborhood**, for example a (3	imes3) window centered on that pixel.

For a pixel at row (y), column (x), a (3	imes3) neighborhood can be written as:

```python
patch = image[y-1:y+2, x-1:x+2]
```

This simple idea is the basis of spatial filtering, smoothing, sharpening, edge detection, and many local statistics.

## 6. Regions of interest (ROI)

A region of interest is simply a slice of the image array. If the image is `image[y1:y2, x1:x2]`, the first interval selects rows and the second selects columns.


### Region of interest (ROI)

An ROI is a selected subregion of an image. It lets us focus computation and visualization on the part that matters.

With NumPy slicing:

```python
roi = image[y_start:y_end, x_start:x_end]
```

Notice again that the vertical range comes first because arrays are indexed as `[row, column]`.

In [8]:
# NumPy slicing uses [row_start:row_end, column_start:column_end].
tower = images["tower"]
h, w, _ = tower.shape

# Central crop occupying half the width and half the height.
y1, y2 = h // 4, 3 * h // 4
x1, x2 = w // 4, 3 * w // 4
roi = tower[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(tower)
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2)
axes[0].add_patch(rect)
axes[0].set_title("Full image and ROI")
axes[1].imshow(roi)
axes[1].set_title(f"ROI: {roi.shape[1]}×{roi.shape[0]}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_region_of_interest.png", dpi=100, bbox_inches="tight")
plt.show()

## 7. RGB channels

Each RGB channel is itself a 2-D image. Displaying channels separately helps reveal which structures contribute strongly to red, green, or blue intensity.


### RGB channels are separate numerical planes

An RGB image can be viewed as three grayscale-like matrices stacked together:

[
I(:,:,0)=R,qquad I(:,:,1)=G,qquad I(:,:,2)=B
]

A pixel such as `[200, 30, 20]` contains much more red than green or blue, so it appears reddish.

Channel analysis is useful in color segmentation, enhancement, feature extraction, and visualization.

### Common library pitfall: RGB vs BGR

Pillow and Matplotlib use **RGB** order.

OpenCV traditionally loads color images as **BGR**.

If an RGB image is interpreted as BGR, red and blue are exchanged and colors look wrong. Always know the channel convention of the library you are using.

In [9]:
# The last axis stores color channels in RGB order when using Pillow.
red = peppers[..., 0]
green = peppers[..., 1]
blue = peppers[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(peppers)
axes[0].set_title("RGB")
for ax, channel, title in zip(axes[1:], [red, green, blue], ["Red channel", "Green channel", "Blue channel"]):
    ax.imshow(channel, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_rgb_channels.png", dpi=100, bbox_inches="tight")
plt.show()

print("Channel means:", {
    "R": round(float(red.mean()), 2),
    "G": round(float(green.mean()), 2),
    "B": round(float(blue.mean()), 2),
})

Channel means: {'R': 140.79, 'G': 132.66, 'B': 146.9}


## 8. Basic image statistics

For a grayscale image, useful first summaries include minimum, maximum, mean, standard deviation, and percentiles. They do not describe spatial arrangement, but they help characterize brightness and contrast.


### What image statistics tell us

Useful scalar summaries include:

- minimum and maximum intensity;
- mean intensity;
- median intensity;
- standard deviation;
- percentiles.

These help describe brightness, spread, and dynamic range.

But statistics do **not** describe where structures occur. Two very different images can have similar means and standard deviations.

In [10]:
# Statistics summarize intensity values but do not describe spatial arrangement.
grass_gray = rgb_to_grayscale(images["grass"])

def image_statistics(gray_image):
    return {
        "min": int(gray_image.min()),
        "max": int(gray_image.max()),
        "mean": float(gray_image.mean()),
        "std": float(gray_image.std()),
        "p05": float(np.percentile(gray_image, 5)),
        "median": float(np.median(gray_image)),
        "p95": float(np.percentile(gray_image, 95)),
    }

stats = image_statistics(grass_gray)
for key, value in stats.items():
    print(f"{key:>6s}: {value:.2f}" if isinstance(value, float) else f"{key:>6s}: {value}")

   min: 0
   max: 255
  mean: 79.51
   std: 50.50
   p05: 11.00
median: 70.00
   p95: 178.00


## 9. Intensity histograms

For 8-bit grayscale data, a histogram counts how many pixels have intensities from 0 to 255.

- a narrow histogram usually indicates low contrast;
- a wide histogram indicates a broader use of the available dynamic range;
- histogram shape alone does **not** tell us where intensities occur spatially.


### What a histogram means

An intensity histogram counts how many pixels fall into each intensity bin.

For an 8-bit grayscale image, we commonly use 256 bins for values 0–255.

A histogram can reveal:

- whether an image is globally dark or bright;
- whether its intensities occupy a narrow or wide range;
- whether there are several intensity populations.

A histogram does **not** contain spatial information. If pixels are rearranged, the image can look completely different while keeping exactly the same histogram.

In [11]:
# A 256-bin histogram gives one bin for each possible uint8 intensity.
ballons_gray = rgb_to_grayscale(images["ballons"])
counts, bin_edges = np.histogram(ballons_gray.ravel(), bins=256, range=(0, 256))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")
axes[1].plot(bin_edges[:-1], counts)
axes[1].set_title("256-bin intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_intensity_histogram.png", dpi=100, bbox_inches="tight")
plt.show()

print("Histogram pixel-count check:", counts.sum(), "==", ballons_gray.size)

Histogram pixel-count check: 180000 == 180000


## 10. Dynamic range and normalization

A simple min-max normalization expands the observed range $[I_{min}, I_{max}]$ to $[0,255]$:

$$I_{norm} = 255\frac{I-I_{min}}{I_{max}-I_{min}}.$$

This can improve contrast when the original image uses only a limited part of the available intensity range.


### Dynamic range and normalization

The **dynamic range** is the span of intensity values used by an image.

If most values occupy only a small interval, the image may look low-contrast.

Min-max normalization maps the observed minimum and maximum to a target range:

[
I_{	ext{norm}} =
rac{I-I_{min}}{I_{max}-I_{min}}
]

For an 8-bit result, this normalized value is then scaled to 0–255.

Normalization changes the numerical range. It is not the same as histogram equalization, which redistributes intensities according to their cumulative distribution.

In [12]:
def minmax_normalize(gray_image):
    """Stretch a grayscale image linearly to the full 8-bit range."""
    arr = gray_image.astype(np.float32)
    low, high = arr.min(), arr.max()
    if high == low:
        return np.zeros_like(gray_image)
    normalized = 255.0 * (arr - low) / (high - low)
    return np.clip(normalized, 0, 255).astype(np.uint8)

# Create a deliberately low-contrast version to make the effect visible.
low_contrast = (90 + 0.30 * ballons_gray).clip(0, 255).astype(np.uint8)
normalized = minmax_normalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 1].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("After min-max normalization")
axes[1, 0].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[1, 0].set_title("Before")
axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After")
for ax in axes[0]:
    ax.axis("off")
for ax in axes[1]:
    ax.set_xlim(0, 255)
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")
fig.tight_layout()
fig.savefig(FIG_DIR / "08_dynamic_range_normalization.png", dpi=100, bbox_inches="tight")
plt.show()

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range:", int(normalized.min()), "to", int(normalized.max()))

Before range: 90 to 166
After range: 0 to 255


## 11. Introducing synthetic noise

Noise is unavoidable in real acquisition systems. Here we add Gaussian noise only to understand its numerical effect; later labs will study filtering methods for reducing it.

For additive Gaussian noise:

$$g(x,y)=f(x,y)+n(x,y), \qquad n\sim\mathcal{N}(0,\sigma^2).$$


### Noise: unwanted variation in image values

Noise can come from sensors, electronics, acquisition conditions, transmission, compression, or modeling assumptions.

Common models include:

| Noise type | Typical behavior |
|---|---|
| Gaussian | additive continuous fluctuations |
| Salt-and-pepper | isolated very dark or very bright pixels |
| Poisson / shot noise | signal-dependent counting noise |
| Speckle | multiplicative granular noise |

Noise matters because later filtering methods are often designed around a specific noise model.

In [13]:
# Add zero-mean Gaussian noise in floating point, then clip to the valid 8-bit range.
sigma = 20.0
noise = RNG.normal(loc=0.0, scale=sigma, size=einstein_gray.shape)
noisy_float = einstein_gray.astype(np.float32) + noise
noisy_einstein = np.clip(noisy_float, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[1].imshow(noise, cmap="gray")
axes[1].set_title(f"Gaussian noise (σ={sigma:.0f})")
axes[2].imshow(noisy_einstein, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Noisy image")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_gaussian_noise.png", dpi=100, bbox_inches="tight")
plt.show()

## 12. Quantifying image differences

Two simple error measures are useful before learning more advanced image-quality metrics.

Mean absolute error:

$$MAE = \frac{1}{N}\sum_i |x_i-y_i|$$

Root mean squared error:

$$RMSE = \sqrt{\frac{1}{N}\sum_i(x_i-y_i)^2}$$

For 8-bit images, PSNR can be computed from MSE as

$$PSNR = 10\log_{10}\left(\frac{255^2}{MSE}\right).$$


### Error metrics for image comparison

Given a reference image (R) and a test image (T):

[
mathrm{MAE}=rac{1}{N}sum |R-T|
]

[
mathrm{MSE}=rac{1}{N}sum (R-T)^2
]

[
mathrm{RMSE}=sqrt{mathrm{MSE}}
]

For 8-bit images, PSNR is commonly defined as

[
mathrm{PSNR}=10log_{10}left(rac{255^2}{mathrm{MSE}}ight)
]

Higher PSNR usually means the test image is numerically closer to the reference.

These metrics are useful, but they do not perfectly model human perception. A small numerical error can be visually important, and a larger one can sometimes be visually harmless.

In [14]:
def comparison_metrics(reference, test):
    ref = reference.astype(np.float64)
    tst = test.astype(np.float64)
    diff = ref - tst
    mae = np.mean(np.abs(diff))
    mse = np.mean(diff ** 2)
    rmse = np.sqrt(mse)
    psnr = np.inf if mse == 0 else 10 * np.log10((255.0 ** 2) / mse)
    return mae, rmse, psnr

mae, rmse, psnr = comparison_metrics(einstein_gray, noisy_einstein)
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"PSNR: {psnr:.2f} dB")

# Reasonableness checks: noise must change the image and metrics must be finite.
assert mae > 0
assert rmse > 0
assert np.isfinite(psnr)

MAE : 15.85
RMSE: 19.85
PSNR: 22.18 dB


## 13. Saving a processed image

A reproducible notebook should save meaningful outputs explicitly rather than relying only on what is visible in memory.


### Image file formats: what you should know

**PNG**
- lossless compression;
- preserves exact pixel values after save/load in normal use;
- good for masks, diagrams, labels, and reproducible intermediate results.

**JPEG**
- lossy compression;
- designed mainly for natural photographs;
- decoded pixel values may differ from the original;
- repeated save/load cycles can introduce artifacts.

For quantitative experiments, prefer lossless formats unless compression itself is part of the experiment.

In [15]:
# Use a lossless PNG file so the saved pixel values remain suitable for experiments.
saved_path = FIG_DIR / "einstein_noisy.png"
Image.fromarray(noisy_einstein).save(saved_path)

reloaded = np.asarray(Image.open(saved_path))
print("Saved:", f"../outputs/figures/{saved_path.name}")
print("Reloaded shape:", reloaded.shape)
print("Exact round-trip equality:", np.array_equal(noisy_einstein, reloaded))
assert np.array_equal(noisy_einstein, reloaded)

Saved: ../outputs/figures/einstein_noisy.png
Reloaded shape: (256, 256)
Exact round-trip equality: True


## 14. Common mistakes to avoid

1. **Confusing `(x, y)` with `[row, column]`.**  
   NumPy indexing is `image[y, x]`.

2. **Assuming every image is RGB.**  
   Check `ndim` and `shape` before indexing channels.

3. **Doing arithmetic directly in `uint8`.**  
   Convert to a wider or floating type first when necessary.

4. **Forgetting to clip before converting back to `uint8`.**

5. **Mixing RGB and BGR conventions.**

6. **Using Matplotlib autoscaling without noticing it.**  
   For grayscale comparisons, explicit `vmin` and `vmax` can be important.

7. **Changing the original array unintentionally.**  
   Use `.copy()` when you need an independent image.

8. **Comparing images of different shapes or data ranges.**

9. **Treating a histogram as spatial information.**

10. **Saving quantitative results as JPEG without considering compression artifacts.**

## 15. Checks

The following checks summarize the essential invariants developed in this lab.


In [16]:
assert peppers.ndim == 3 and peppers.shape[2] == 3
assert einstein_gray.ndim == 2
assert einstein_gray.dtype == np.uint8
assert 0 <= einstein_gray.min() <= einstein_gray.max() <= 255
assert counts.sum() == ballons_gray.size
assert normalized.min() == 0 and normalized.max() == 255

expected_figures = [
    "01_grayscale_matrix.png",
    "02_dataset_overview.png",
    "03_rgb_to_grayscale.png",
    "04_pixel_edit.png",
    "05_region_of_interest.png",
    "06_rgb_channels.png",
    "07_intensity_histogram.png",
    "08_dynamic_range_normalization.png",
    "09_gaussian_noise.png",
    "einstein_noisy.png",
]
missing_figures = [name for name in expected_figures if not (FIG_DIR / name).exists()]
assert not missing_figures, f"Missing outputs: {missing_figures}"

print("All fundamental checks passed.")
print(f"Generated {len(expected_figures)} output figures/images.")

All fundamental checks passed.
Generated 10 output figures/images.


## 16. Practical exercises

Try these without changing the reference cells above:

1. **Coordinates:** choose three pixels in `peppers.png`, print their `(R,G,B)` values, and locate them visually.
2. **Grayscale:** compare the luminance formula with a simple channel average `(R+G+B)/3`. Where are the largest differences?
3. **ROI:** extract a different ROI from `Elizabeth_Tower_London.jpg` and report its dimensions.
4. **Histogram:** compare the grayscale histograms of `grass.jpg` and `ballons.jpg`. Which uses a wider intensity range?
5. **Noise:** repeat the noise experiment with $\sigma=5$, $20$, and $50$. Track MAE, RMSE, and PSNR.
6. **Normalization:** construct a low-contrast image with another linear transform and verify that min-max normalization uses the full 0–255 range.

**Common mistakes to avoid**

- confusing `(x, y)` with NumPy `[row, column]`;
- modifying the original array when a copy was intended;
- performing arithmetic directly on `uint8` when values can overflow or underflow;
- displaying grayscale data without `cmap="gray"`;
- assuming a histogram contains spatial information.


### Suggested beginner workflow for every new image

Whenever you receive an unfamiliar image, inspect it in this order:

```text
1. Load
2. Check shape
3. Check dtype
4. Check min / max
5. Display
6. Identify channel convention
7. Inspect statistics
8. Decide whether conversion or normalization is needed
9. Process
10. Validate the result numerically and visually
```

This workflow prevents many silent errors.

### Mini-checkpoint questions

Before moving on, you should be able to answer these without looking up the notebook:

1. Why is an RGB image usually a 3-D array?
2. Why does `image[100, 200]` mean row 100, column 200?
3. What is the difference between sampling and quantization?
4. What information is lost when converting RGB to grayscale?
5. Why can `uint8` arithmetic be dangerous?
6. What does a histogram tell you, and what does it not tell you?
7. Why is a region of interest useful?
8. What is the difference between a pixel and a pixel neighborhood?
9. Why is PNG preferable to JPEG for many quantitative experiments?
10. Why should both numerical metrics and visual inspection be used when comparing images?

## 17. Key takeaways

- A digital image is a numerical array; visualization is only one representation of that data.
- Grayscale images are 2-D arrays, while RGB images usually have three channels.
- `uint8` images represent each channel with values from 0 to 255.
- Pixel and ROI operations are NumPy indexing operations.
- RGB channels carry different information and can be studied independently.
- Histograms describe the distribution of intensities but not their spatial arrangement.
- Normalization changes the mapping of intensity values; it does not recover information that was never captured.
- Controlled noise experiments provide a foundation for the next lab on filtering.

**Next lab:** `Image_Transformation`.


## Glossary

| Term | Meaning |
|---|---|
| Pixel | smallest sampled spatial element of a digital image |
| Intensity | numerical brightness value |
| Sampling | discretization of spatial coordinates |
| Quantization | discretization of intensity/amplitude values |
| Resolution | amount of spatial detail represented; context must be specified |
| Channel | one component plane of an image, such as R, G, or B |
| Bit depth | number of bits used to encode an intensity/channel value |
| Dynamic range | span of represented intensity values |
| ROI | region of interest selected for focused analysis |
| Histogram | distribution of pixel intensities |
| Noise | unwanted variation in pixel measurements |
| PSNR | logarithmic error-based similarity measure relative to a reference |

## Where this leads next

After this notebook, the next labs can be understood as extensions of the same array model:

- **Image Transformation** — change pixel intensities or spatial coordinates;
- **Filtering in Spatial Domain** — combine values from local neighborhoods;
- **Filtering in Frequency Domain** — manipulate spatial-frequency content;
- **Image Segmentation** — assign pixels or regions to meaningful classes.

The important transition is this: you should now see an image simultaneously as **visual content**, **a matrix**, and **measured data**.